In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.neighbors import NearestNeighbors

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [2]:
with open('../../data/nifty_ts2vec_embeddings.pkl', 'rb') as f:
    df = pickle.load(f)

In [3]:
df

,0,1,2,3,4,5,6,7,8,9,...,55,56,57,58,59,60,61,62,63,Close
Date,,,,,,,,,,,,,,,,,,,,,
2008-06-05 00:00:00+05:30,0.000000,0.094707,0.000000,0.086095,0.000000,0.052627,0.000000,0.034146,0.049268,0.079448,...,0.000000,0.007209,0.042414,0.000000,0.000000,0.097714,0.014177,0.101710,0.000000,4676.950195
2008-06-06 00:00:00+05:30,0.000000,0.096463,0.000000,0.093592,0.000000,0.055477,0.000000,0.038427,0.048164,0.072977,...,0.000000,0.004996,0.039518,0.000000,0.000000,0.092051,0.016242,0.101359,0.000000,4627.799805
2008-06-09 00:00:00+05:30,0.000000,0.097814,0.000000,0.100934,0.000000,0.060673,0.000000,0.042575,0.046727,0.065568,...,0.000000,0.002542,0.034720,0.000000,0.000000,0.085397,0.017465,0.100786,0.000000,4500.950195
2008-06-10 00:00:00+05:30,0.000000,0.099855,0.000000,0.104100,0.000000,0.071059,0.000000,0.047309,0.046517,0.058178,...,0.000000,0.001713,0.030261,0.000000,0.000000,0.080686,0.019027,0.099659,0.000000,4449.799805
2008-06-11 00:00:00+05:30,0.000000,0.100483,0.000000,0.105155,0.000000,0.084748,0.000000,0.052642,0.045687,0.050810,...,0.000058,0.000870,0.026759,0.000000,0.000000,0.075993,0.020184,0.097661,0.000000,4523.600098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-12 00:00:00+05:30,0.150253,0.171876,0.110617,0.016477,0.000274,0.000000,0.104171,0.077467,0.020435,0.000000,...,0.073654,0.000000,0.000000,0.025725,0.050502,0.000000,0.019644,0.066028,0.036041,25114.000000
2025-09-15 00:00:00+05:30,0.140572,0.171512,0.102708,0.012511,0.000518,0.000000,0.113352,0.073049,0.019066,0.000000,...,0.072866,0.000000,0.000000,0.025318,0.053611,0.000000,0.021749,0.061992,0.034028,25069.199219
2025-09-16 00:00:00+05:30,0.128521,0.175892,0.093519,0.008835,0.000000,0.000000,0.121123,0.067323,0.016315,0.000000,...,0.075898,0.000000,0.000000,0.024032,0.058974,0.000000,0.025540,0.057928,0.034247,25239.099609


In [4]:
def embedding_stability_score(
    embeddings,
    var_window=20,
    eps=1e-8
):
    """
    embeddings: np.ndarray of shape (T, D)
    returns: pd.Series of length T in (0, 1]
    """

    T, D = embeddings.shape

    # ---- Velocity: ||z_t - z_{t-1}||
    velocity = np.zeros(T)
    velocity[1:] = np.linalg.norm(
        embeddings[1:] - embeddings[:-1], axis=1
    )

    # ---- Local variance over rolling window
    # Rolling variance across time, then average over dims
    emb_df = pd.DataFrame(embeddings)
    rolling_var = emb_df.rolling(var_window).var().mean(axis=1).values

    # ---- Normalize (robust)
    vel_norm = velocity / (np.median(velocity) + eps)
    var_norm = rolling_var / (np.median(rolling_var[var_window:]) + eps)

    # ---- Stability score (higher = more stable)
    instability = vel_norm + var_norm
    stability = np.exp(-instability)

    return pd.Series(stability, name="embedding_stability")


def familiarity_score_fn(
    embeddings,
    lookback=60,
    eps=1e-8
):
    """
    embeddings: np.ndarray of shape (T, D)
    returns: pd.Series of length T in (0, 1]
    """

    T, D = embeddings.shape
    distances = np.zeros(T)

    for t in range(T):
        if t < lookback:
            distances[t] = np.nan
            continue

        recent_mean = embeddings[t - lookback:t].mean(axis=0)
        distances[t] = np.linalg.norm(
            embeddings[t] - recent_mean
        )

    # ---- Normalize robustly
    dist_norm = distances / (np.nanmedian(distances) + eps)

    # ---- Familiarity score (higher = more familiar)
    familiarity = np.exp(-dist_norm)

    return pd.Series(familiarity, name="embedding_familiarity")

def familiarity_score_fn_mahalanobis(
    embeddings,
    lookback=60,
    eps=1e-6,
    shrinkage=0.05
):
    """
    embeddings: np.ndarray of shape (T, D)
    returns: pd.Series of length T in (0, 1]
    """

    T, D = embeddings.shape
    distances = np.full(T, np.nan)

    for t in range(lookback, T):
        window = embeddings[t - lookback:t]

        # ---- Local mean
        mu = window.mean(axis=0)

        # ---- Local covariance
        cov = np.cov(window, rowvar=False)

        # ---- Shrinkage for numerical stability
        cov = (1 - shrinkage) * cov + shrinkage * np.eye(D)

        # ---- Invert covariance
        try:
            inv_cov = np.linalg.inv(cov)
        except np.linalg.LinAlgError:
            inv_cov = np.linalg.pinv(cov)

        diff = embeddings[t] - mu

        # ---- Mahalanobis distance
        distances[t] = np.sqrt(diff.T @ inv_cov @ diff)

    # ---- Robust normalization
    median = np.nanmedian(distances)
    dist_norm = distances / (median + eps)

    # ---- Familiarity score
    # familiarity = np.exp(-dist_norm)
    familiarity = np.exp(-0.5 * dist_norm)

    return pd.Series(familiarity, name="embedding_familiarity")

def density_score_knn(
    embeddings,
    k=20
):
    """
    embeddings: np.ndarray (T, D)
    returns: pd.Series (T,)
    """

    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(embeddings)
    distances, _ = nbrs.kneighbors(embeddings) 

    # Ignore self-distance
    knn_dist = distances[:, 1:].mean(axis=1)

    knn_norm = knn_dist / np.median(knn_dist)
    density = np.exp(-knn_norm)

    return pd.Series(density, name="embedding_density")


In [ ]:
# stability_score = embedding_stability_score(
#     embeddings=df.drop(columns=["Cluster","Close"]).values,
#     var_window=20
# )

# familiarity_score = familiarity_score_fn(
#     embeddings=df.drop(columns=["Cluster","Close"]).values,
#     lookback=60
# )

# density_score = density_score_knn(
#     embeddings=df.drop(columns=["Cluster","Close"]).values,
#     k=20
# )

# bull_confidence = stability_score * familiarity_score


In [ ]:
# sdf=pd.DataFrame(
#     {
#         "Date": df.index,
#         "bull_confidence": bull_confidence,
#         "Close": df["Close"].values
#     }
# )
# sdf=sdf.set_index("Date")

In [ ]:
# sdf["Close_Scaled"] = (sdf["Close"] - sdf["Close"].min()) / (sdf["Close"].max() - sdf["Close"].min())
# sdf

,bull_confidence,Close,Close_Scaled
Date,,,
2021-09-03 00:00:00+05:30,NaN,17323.599609,0.185863
2021-09-06 00:00:00+05:30,NaN,17377.800781,0.190825
2021-09-07 00:00:00+05:30,NaN,17362.099609,0.189388
2021-09-08 00:00:00+05:30,NaN,17353.500000,0.188601
2021-09-09 00:00:00+05:30,NaN,17369.250000,0.190043
...,...,...,...
2025-09-12 00:00:00+05:30,0.037748,25114.000000,0.899103
2025-09-15 00:00:00+05:30,0.036008,25069.199219,0.895001
2025-09-16 00:00:00+05:30,0.036732,25239.099609,0.910556


In [ ]:
def build_df(dataframe, start_date, end_date):
    mask = (dataframe.index >= start_date) & (dataframe.index <= end_date)
    tdf = dataframe.loc[mask].copy()

    stability_score = embedding_stability_score(
        embeddings=tdf.drop(columns=["Close"]).values,
        var_window=20
    )

    familiarity_score = familiarity_score_fn_mahalanobis(
        embeddings=tdf.drop(columns=["Close"]).values,
        lookback=40,
        shrinkage=0.05
    )

    # density_score = density_score_knn(
    #     embeddings=tdf.drop(columns=["Cluster","Close"]).values,
    #     k=20
    # )

    bull_confidence = stability_score * familiarity_score
    
    sdf=pd.DataFrame(
        {
            "Date": tdf.index,
            "bull_confidence": bull_confidence,
            "Close": tdf["Close"].values
        }
    )
    sdf=sdf.set_index("Date")

    sdf["Close_Scaled"] = (sdf["Close"] - sdf["Close"].min()) / (sdf["Close"].max() - sdf["Close"].min())

    
    sdf["bull_confidence"] = sdf["bull_confidence"].ewm(span=5).mean()
    sdf["eligible"] = sdf["bull_confidence"] > sdf["bull_confidence"].rolling(252).quantile(0.6)
    sdf["bull_trend"] = sdf["bull_confidence"].diff(5)
    sdf["bull_z"] = (
        sdf["bull_confidence"] - sdf["bull_confidence"].rolling(252).mean()
    ) / sdf["bull_confidence"].rolling(252).std()

    sdf["trend_ok"] = sdf["Close"] > sdf["Close"].rolling(50).mean()

    sdf["enter_long"] = (sdf["bull_z"] > 0.5) &  (sdf["bull_z"].diff(5) > 0) & sdf["trend_ok"]
    
    return sdf

tdf = build_df(df, "2022-01-01", "2024-01-05")
tdf.tail(10)

,bull_confidence,Close,Close_Scaled,eligible,bull_trend,bull_z,trend_ok,enter_long
Date,,,,,,,,
2023-12-22 00:00:00+05:30,0.074278,21349.400391,0.933803,False,0.005751,-0.285582,True,False
2023-12-26 00:00:00+05:30,0.076479,21441.349609,0.947982,False,0.004835,-0.181593,True,False
2023-12-27 00:00:00+05:30,0.077414,21654.750000,0.980887,False,0.005100,-0.140688,True,False
2023-12-28 00:00:00+05:30,0.078350,21778.699219,1.000000,False,0.006549,-0.099784,True,False
2023-12-29 00:00:00+05:30,0.080168,21731.400391,0.992707,False,0.006512,-0.014524,True,False
2024-01-01 00:00:00+05:30,0.083978,21741.900391,0.994326,True,0.009700,0.173073,True,False
2024-01-02 00:00:00+05:30,0.084702,21665.800781,0.982591,True,0.008223,0.203008,True,False
2024-01-03 00:00:00+05:30,0.086234,21517.349609,0.959701,True,0.008820,0.275281,True,False
2024-01-04 00:00:00+05:30,0.085164,21658.599609,0.981481,True,0.006814,0.212020,True,False


In [18]:
tdf.tail(10)

,bull_confidence,Close,Close_Scaled,eligible,bull_trend,bull_z,trend_ok,enter_long
Date,,,,,,,,
2023-12-22 00:00:00+05:30,0.074278,21349.400391,0.933803,False,0.005751,-0.285582,True,False
2023-12-26 00:00:00+05:30,0.076479,21441.349609,0.947982,False,0.004835,-0.181593,True,False
2023-12-27 00:00:00+05:30,0.077414,21654.750000,0.980887,False,0.005100,-0.140688,True,False
2023-12-28 00:00:00+05:30,0.078350,21778.699219,1.000000,False,0.006549,-0.099784,True,False
2023-12-29 00:00:00+05:30,0.080168,21731.400391,0.992707,False,0.006512,-0.014524,True,False
2024-01-01 00:00:00+05:30,0.083978,21741.900391,0.994326,True,0.009700,0.173073,True,False
2024-01-02 00:00:00+05:30,0.084702,21665.800781,0.982591,True,0.008223,0.203008,True,False
2024-01-03 00:00:00+05:30,0.086234,21517.349609,0.959701,True,0.008820,0.275281,True,False
2024-01-04 00:00:00+05:30,0.085164,21658.599609,0.981481,True,0.006814,0.212020,True,False


In [19]:
# sdf.drop(columns=["Close"]).tail(500).plot(secondary_y="Close_Scaled", figsize=(12,6))

In [65]:

def add_boolean_colored_line(fig, x, y, flag, colors, name):
    """
    fig: plotly fig
    x: index or array
    y: values
    flag: boolean array
    colors: {True: color, False: color}
    """

    start = 0
    for i in range(1, len(flag)):
        if flag[i] != flag[i - 1]:
            fig.add_trace(
                go.Scatter(
                    x=x[start:i],
                    y=y.iloc[start:i],
                    mode="lines",
                    line=dict(color=colors[flag.iloc[start]]),
                    name=name if start == 0 else None,
                    showlegend=start == 0,
                )
            )
            start = i

    # last segment
    fig.add_trace(
        go.Scatter(
            x=x[start:],
            y=y.iloc[start:],
            mode="lines",
            line=dict(color=colors[flag.iloc[start]]),
            name=name if start == 0 else None,
            showlegend=start == 0,
        )
    )


def plot_score_vs_price(dataframe):
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # fig.add_trace(
    #     go.Scatter(
    #         x=dataframe.index,
    #         y=dataframe["bull_confidence"],
    #         name="Bull Market Confidence",
    #         line=dict(color='blue')
    #     ),
    #     secondary_y=False,
    # )
    fig.add_trace(
        go.Scatter(
            x=dataframe.index,
            y=dataframe["bull_z"],
            name="Bull Market Confidence Z-Score",
            line=dict(color='red')
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=dataframe.index,
            y=dataframe["Close_Scaled"],
            name="NIFTY Close Price",
            line=dict(color='orange'),
            
        ),
        secondary_y=True,
    )

    # add_boolean_colored_line(
    #     fig,
    #     x=dataframe.index,
    #     y=dataframe["bull_confidence"],
    #     flag=dataframe["enter_long"],
    #     colors={True: "green", False: "red"},
    #     name="Bull Confidence",
    # )




    fig.update_layout(
        title="NIFTY Embedding-based Bull Market Confidence Score vs Close Price",
        xaxis_title="Date",
        yaxis_title="Bull Market Confidence",
        legend_title="Legend",
    )
    fig.update_yaxes(title_text="NIFTY Close Price", secondary_y=True)  
    return fig


In [68]:
start_date = "2023-05-01"
end_date = "2026-01-01"
sdf = build_df(df, start_date, end_date)
plot_score_vs_price(sdf).show()

In [58]:
sdf["eligible"].value_counts()

eligible
False    882
True     453
Name: count, dtype: int64

In [32]:
start_date = "2024-05-01"
end_date = "2026-01-01"
sdf = build_df(df, start_date, end_date)
plot_score_vs_price(sdf).show()

In [34]:
start_date = "2020-01-01"
end_date = "2022-01-01"
sdf = build_df(df, start_date, end_date)
plot_score_vs_price(sdf).show()

In [20]:
sdf[["Close","bull_confidence"]].describe().to_clipboard()

In [ ]:
	bull_confidence	Close	Close_Scaled
count	438.0	498.0	498.0
mean	0.053512811762533447	13580.09275774975	0.5493652527476484
std	0.020015782058013194	2787.6387731355717	0.2565280094161174
min	0.015090117426150157	7610.25	0.0
25%	0.04001427906824301	11318.63720703125	0.3412584146596158
50%	0.05205475320955627	13902.89990234375	0.579071065073847
75%	0.0629567594602635	15760.387451171875	0.7500033924643617
max	0.11859691649654304	18477.05078125	1.0
